Data Import

In [1]:
import pandas as pd
from pathlib import Path



df = pd.read_csv(Path.cwd().parent / "data" / "csvs" / "first_5_days_montea_and_price.csv")
df["price"] = df["price [€/MWh]"]


packages 


In [2]:
from ortools.init.python import init
from ortools.linear_solver import pywraplp
import plotly.graph_objects as go
from plotly.subplots import make_subplots


BESS dispatch optimizer — day-ahead price arbitrage.
 
Model (v1 — minimal, as agreed):
  - Perfect round-trip efficiency
  - No charge/discharge power limits
  - Only constraint: battery energy capacity (SoC in [0, capacity])
  - Site offtake/injection are fixed (inflexible) profiles that the battery
    sits "behind the meter" with, netting against the grid
  - Objective: minimize total cost of net grid exchange at the DA price
    (offtake costs price, injection is credited at the same price — i.e.
    net metering / symmetric spot exposure)
 
This is a linear program (no binaries needed): since buy and sell use the
same price in the same interval, the LP will never charge and discharge
simultaneously in an optimal solution (doing so is cost-neutral at best),
so solving with GLOP (ortools.linear_solver) is sufficient and fast.
 
Assumptions made explicit here (change these constants to match reality):
  - CAPACITY_MWH: battery energy capacity
  - INITIAL_SOC_MWH: starting state of charge (assumed empty)
  - FINAL_SOC_MWH: if set, forces ending SoC (None = free, battery can end
    anywhere within capacity)

Assumptions

In [3]:
# ----------------------------- Assumptions ----------------------------- #
CAPACITY_MWH = 2.0      # battery energy capacity
INITIAL_SOC_MWH = 0.0   # battery starts empty
DT_HOURS = 0.25         # fixed 15-minute timestep

In [4]:
df["off_mwh"] = df["off"] * DT_HOURS / 1000.0   # kW -> MWh for this interval
df["inj_mwh"] = df["inj"] * DT_HOURS / 1000.0
print(df.head())
n = len(df)

total_energy_mwh = df["off_mwh"].sum() - df["inj_mwh"].sum()
print(f"Net energy offtake in the 5 days: {total_energy_mwh:.2f} MWh")

                       dates  con  gen       off  inj  price [€/MWh]  \
0  2024-12-31 23:00:00+00:00  NaN  NaN   0.19325  0.0          10.62   
1  2024-12-31 23:15:00+00:00  NaN  NaN   2.82500  0.0          10.62   
2  2024-12-31 23:30:00+00:00  NaN  NaN  17.40000  0.0          10.62   
3  2024-12-31 23:45:00+00:00  NaN  NaN   6.62500  0.0          10.62   
4  2025-01-01 00:00:00+00:00  NaN  NaN   9.40000  0.0          10.27   

   cleared_volume [MW]  price   off_mwh  inj_mwh  
0              742.525  10.62  0.000048      0.0  
1              742.525  10.62  0.000706      0.0  
2              742.525  10.62  0.004350      0.0  
3              742.525  10.62  0.001656      0.0  
4              695.600  10.27  0.002350      0.0  
Net energy offtake in the 5 days: 0.41 MWh


In [5]:
# ------------------------------ Build the LP ------------------------------ #
solver = pywraplp.Solver.CreateSolver("GLOP")

 #decision variables
charge = [solver.NumVar(0, solver.infinity(), f"charge_{t}") for t in range(n)]
discharge = [solver.NumVar(0, solver.infinity(), f"discharge_{t}") for t in range(n)]
soc = [solver.NumVar(0, CAPACITY_MWH, f"soc_{t}") for t in range(n)]


 #constraints
for t in range(n):
    prev_soc = INITIAL_SOC_MWH if t == 0 else soc[t - 1]
    solver.Add(soc[t] == prev_soc + charge[t] - discharge[t])
 
# cost[t] = price[t] * (off[t] - inj[t] + charge[t] - discharge[t]) where cost is the objective function we want to minimize
# (off - inj) is a fixed number, not a variable, so it doesn't change where
# the optimum is -- it only affects the total cost we report afterwards
objective = solver.Objective()
for t in range(n):
    price = df["price"].iloc[t]       #iloc is used to access the value at index t in the dataframe not the same as loc which is used to access the value at a specific label in the dataframe
    objective.SetCoefficient(charge[t], price)  #increasing charge increases objective by price, so we want to minimize charge when price is high
    objective.SetCoefficient(discharge[t], -price) #increasing discharge decreases objective by price, so we want to maximize discharge when price is high
objective.SetMinimization()
 
solver.Solve()
 
# -------------------------------- Results --------------------------------- #
df["charge_mwh"] = [v.solution_value() for v in charge]
df["discharge_mwh"] = [v.solution_value() for v in discharge]
df["soc_mwh"] = [v.solution_value() for v in soc]
 
df["net_grid_baseline"] = df["off_mwh"] - df["inj_mwh"]
df["net_grid_with_bess"] = df["net_grid_baseline"] + df["charge_mwh"] - df["discharge_mwh"]
 
baseline_cost = (df["net_grid_baseline"] * df["price"]).sum()
bess_cost = (df["net_grid_with_bess"] * df["price"]).sum()
 
print(f"Baseline cost (no BESS):   € {baseline_cost:,.2f}")
print(f"Cost with BESS:            € {bess_cost:,.2f}")
print(f"Savings from arbitrage:    € {baseline_cost - bess_cost:,.2f}")
 
df.to_csv("bess_dispatch_result.csv", index=False)


#without BESS
cost_offtake = (df["off_mwh"] * df["price"]).sum()
revenue_injection = (df["inj_mwh"] * df["price"]).sum()
print(f"Cost of offtake without BESS: € {cost_offtake:,.2f}")
print(f"Revenue from injection without BESS: € {revenue_injection:,.2f}")

Baseline cost (no BESS):   € -6.46
Cost with BESS:            € -983.46
Savings from arbitrage:    € 977.00
Cost of offtake without BESS: € 188.15
Revenue from injection without BESS: € 194.61


In [6]:
fig = make_subplots(
    rows=3, cols=1,
    shared_xaxes=True,
    row_heights=[0.35, 0.35, 0.3],
    vertical_spacing=0.06,
    subplot_titles=("Day-ahead price (EUR/MWh)", "Battery state of charge (MWh)", "Charge / discharge (MWh per interval)"),
)
 
# --- price ---
fig.add_trace(
    go.Scatter(x=df["dates"], y=df["price"], mode="lines", name="Price",
               line=dict(color="#2a78d6", width=2, shape="hv"),
               fill="tozeroy", fillcolor="rgba(42,120,214,0.08)"),
    row=1, col=1,
)
 
# --- state of charge ---
fig.add_trace(
    go.Scatter(x=df["dates"], y=df["soc_mwh"], mode="lines", name="SoC",
               line=dict(color="#1baf7a", width=2, shape="hv"),
               fill="tozeroy", fillcolor="rgba(27,175,122,0.12)"),
    row=2, col=1,
)
 
# --- charge (positive) / discharge (negative, for visual contrast) ---
fig.add_trace(
    go.Bar(x=df["dates"], y=df["charge_mwh"], name="Charge", marker_color="#1baf7a"),
    row=3, col=1,
)
fig.add_trace(
    go.Bar(x=df["dates"], y=-df["discharge_mwh"], name="Discharge", marker_color="#e34948"),
    row=3, col=1,
)
 
fig.update_layout(
    height=700,
    showlegend=False,
    bargap=0,
    margin=dict(t=40, r=20, l=50, b=40),
    template="plotly_white",
)
fig.update_yaxes(title_text="EUR/MWh", row=1, col=1)
fig.update_yaxes(title_text="MWh", range=[0, 2], row=2, col=1)
fig.update_yaxes(title_text="MWh", row=3, col=1)
 
fig.show()
fig.show(renderer="browser")  # uncomment to open in browser
# fig.write_html("bess_dispatch.html")  # uncomment to save as a standalone interactive file
 